# LAB 5: Tiền xử lý văn bản từ PDF (P2)

**Học phần:** Dữ liệu lớn (Big Data)  
**Mã số sinh viên:** 26410022

Bài này đọc một trang PDF, tìm và thay đoạn văn bản, tách câu/từ, bỏ stop words, stemming và lemmatization. Phần bài tập áp dụng cùng quy trình cho một PDF khác.

## 1. Chuẩn bị thư viện và dữ liệu

Chỉ cài `pypdf`, `nltk` khi môi trường chưa có. Tải ba tài nguyên NLTK cần thiết (`punkt_tab`, `stopwords`, `wordnet`), tránh `nltk.download('all')`. PDF sách trong `datasets/lab5/` được lấy từ liên kết Google Drive của đề.

In [1]:
import importlib.util
import subprocess
import sys
for module, package in [("pypdf", "pypdf"), ("nltk", "nltk")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import re
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize
from pypdf import PdfReader

resources = {
    "punkt_tab": "tokenizers/punkt_tab/english/",
    "stopwords": "corpora/stopwords",
    "wordnet": "corpora/wordnet.zip",
}
for package, resource in resources.items():
    try:
        nltk.data.find(resource)
    except LookupError:
        if not nltk.download(package, quiet=True):
            raise RuntimeError(f"Không tải được NLTK {package}")
print("Đã sẵn sàng: pypdf, NLTK và các tài nguyên cần dùng.")

Đã sẵn sàng: pypdf, NLTK và các tài nguyên cần dùng.


In [2]:
RAW_BASE = "https://raw.githubusercontent.com/shizziio/uit-DS200.G11.CN2.TTNT/main"

def open_pdf(relative_path):
    candidates = [Path(relative_path), Path("..") / relative_path]
    local = next((p for p in candidates if p.is_file()), None)
    if local:
        print("Đọc:", local)
        return PdfReader(local)
    url = f"{RAW_BASE}/{relative_path}"
    print("Đọc:", url)
    return PdfReader(BytesIO(urlopen(url).read()))

book_reader = open_pdf("datasets/lab5/book.pdf")
book_page = book_reader.pages[10].extract_text() or ""  # Trang thứ 11 của PDF
assert book_page.strip(), "Trang sách không chứa văn bản có thể trích xuất"
print("Số trang sách:", len(book_reader.pages), "; ký tự trên trang 11:", len(book_page))
print("Đầu trang:", repr(book_page[:100]))

Đọc: datasets/lab5/book.pdf
Số trang sách: 234 ; ký tự trên trang 11: 2106
Đầu trang: '1\nIntroduction\nOver the past two decades Machine Learning has become one of the main-\nstays of infor'


## 2. Chuyển chữ thường, tìm, thay và cắt văn bản

Đề dùng `find('over')` để bỏ tiêu đề đầu trang, sau đó bỏ các tiêu đề con và dừng tại cụm `to achieve this goal`. Dùng vị trí của chính các cụm từ để cách cắt vẫn đúng khi thư viện trích PDF thay đổi khoảng trắng. Ghép từ bị ngắt ở cuối dòng như `main-\nstays`.

In [3]:
lower = book_page.casefold()
over_index = lower.find("over")
assert over_index >= 0
print("Vị trí đầu tiên của 'over':", over_index)
excerpt = lower[over_index:]
excerpt = re.sub(r"1\.1 a taste of machine learning|1\.1\.1 applications", " ", excerpt)
goal_index = excerpt.find("to achieve this goal")
assert goal_index >= 0
excerpt = excerpt[:goal_index]
excerpt = re.sub(r"(?<=\w)-\s*\n(?=\w)", "", excerpt)
excerpt = re.sub(r"\s+", " ", excerpt).strip()
print("Vị trí mốc cuối:", goal_index, "; ký tự còn lại:", len(excerpt))
print("Đoạn đầu:", repr(excerpt[:110]))

Vị trí đầu tiên của 'over': 15
Vị trí mốc cuối: 1987 ; ký tự còn lại: 1978
Đoạn đầu: 'over the past two decades machine learning has become one of the mainstays of information technology and with '


## 3. Tách câu và tách từ

`sent_tokenize` chia theo câu; `word_tokenize` trả cả dấu câu. Giữ riêng các token chữ cái để bỏ stop words và biến đổi dạng từ.

In [4]:
sentences = sent_tokenize(excerpt)
tokens = word_tokenize(excerpt)
words = [token for token in tokens if token.isalpha()]
print("Số câu:", len(sentences), "; số token:", len(tokens), "; số từ:", len(words))
print("Câu đầu:", sentences[0])
print("20 token đầu:", tokens[:20])

Số câu: 16 ; số token: 369 ; số từ: 334
Câu đầu: over the past two decades machine learning has become one of the mainstays of information technology and with that, a rather central, albeit usually hidden, part of our life.
20 token đầu: ['over', 'the', 'past', 'two', 'decades', 'machine', 'learning', 'has', 'become', 'one', 'of', 'the', 'mainstays', 'of', 'information', 'technology', 'and', 'with', 'that', ',']


## 4. Stop words, stemming và lemmatization

Stop words tiếng Anh được bỏ bằng tập từ của NLTK. `SnowballStemmer` cắt hậu tố; `WordNetLemmatizer` tra từ điển. Ở đây dùng `pos='v'` như ảnh trong đề để minh họa quy về dạng động từ. Với danh từ, có thể dùng `pos='n'`.

In [5]:
english_stopwords = set(stopwords.words("english"))
filtered = [token for token in words if token not in english_stopwords]
stemmer = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()
stems = [stemmer.stem(token) for token in filtered]
lemmas = [lemmatizer.lemmatize(token, pos="v") for token in filtered]
print("Từ sau bỏ stop words:", len(filtered))
print("15 từ đầu:", filtered[:15])
print("15 stem đầu:", stems[:15])
print("15 lemma đầu:", lemmas[:15])

Từ sau bỏ stop words: 178
15 từ đầu: ['past', 'two', 'decades', 'machine', 'learning', 'become', 'one', 'mainstays', 'information', 'technology', 'rather', 'central', 'albeit', 'usually', 'hidden']
15 stem đầu: ['past', 'two', 'decad', 'machin', 'learn', 'becom', 'one', 'mainstay', 'inform', 'technolog', 'rather', 'central', 'albeit', 'usual', 'hidden']
15 lemma đầu: ['past', 'two', 'decades', 'machine', 'learn', 'become', 'one', 'mainstays', 'information', 'technology', 'rather', 'central', 'albeit', 'usually', 'hide']


## 5. Bài tập: áp dụng cho một sách học thuật khác

Dùng *Think Stats, 2nd Edition* của Allen B. Downey, tải từ [Green Tea Press](https://greenteapress.com/wp/think-stats-2e/) theo giấy phép CC BY-NC-SA 4.0. Trang PDF thứ 21 mở đầu chương “Exploratory data analysis”. Lấy đoạn từ “The thesis of this book is” đến trước “Reports like these are called”, rồi áp dụng lại các bước tách câu/từ, bỏ stop words, stemming và lemmatization.

In [6]:
other_reader = open_pdf("datasets/lab5/thinkstats2.pdf")
other_page = other_reader.pages[20].extract_text() or ""  # Trang PDF thứ 21
start = other_page.find("The thesis of this book is")
end = other_page.find("Reports like these are called", start)
assert start >= 0 and end > start
other_excerpt = re.sub(r"(?<=\w)-\s*\n(?=\w)", "", other_page[start:end])
other_excerpt = re.sub(r"\s+", " ", other_excerpt).strip().casefold()
other_sentences = sent_tokenize(other_excerpt)
other_tokens = word_tokenize(other_excerpt)
other_words = [token for token in other_tokens if token.isalpha()]
other_filtered = [token for token in other_words if token not in english_stopwords]
other_stems = [stemmer.stem(token) for token in other_filtered]
other_lemmas = [lemmatizer.lemmatize(token, pos="v") for token in other_filtered]
print("Think Stats: trang PDF 21;", len(other_sentences), "câu;", len(other_tokens), "token;")
print(len(other_filtered), "từ sau bỏ stop words")
print("15 token đầu:", other_tokens[:15])
print("15 từ đã lọc:", other_filtered[:15])
print("15 stem:", other_stems[:15])
print("15 lemma:", other_lemmas[:15])

Đọc: datasets/lab5/thinkstats2.pdf


Think Stats: trang PDF 21; 8 câu; 209 token;
92 từ sau bỏ stop words
15 token đầu: ['the', 'thesis', 'of', 'this', 'book', 'is', 'that', 'data', 'combined', 'with', 'practical', 'methods', 'can', 'answer', 'questions']
15 từ đã lọc: ['thesis', 'book', 'data', 'combined', 'practical', 'methods', 'answer', 'questions', 'guide', 'decisions', 'uncertainty', 'example', 'present', 'case', 'study']
15 stem: ['thesi', 'book', 'data', 'combin', 'practic', 'method', 'answer', 'question', 'guid', 'decis', 'uncertainti', 'exampl', 'present', 'case', 'studi']
15 lemma: ['thesis', 'book', 'data', 'combine', 'practical', 'methods', 'answer', 'question', 'guide', 'decisions', 'uncertainty', 'example', 'present', 'case', 'study']


In [7]:
for word in ["walking", "went", "mice"]:
    print(word, "→ stem:", stemmer.stem(word),
          "; lemma (verb):", lemmatizer.lemmatize(word, pos="v"),
          "; lemma (noun):", lemmatizer.lemmatize(word, pos="n"))

walking → stem: walk ; lemma (verb): walk ; lemma (noun): walking
went → stem: went ; lemma (verb): go ; lemma (noun): went
mice → stem: mice ; lemma (verb): mice ; lemma (noun): mouse


**Nhận xét:** Stemming có thể tạo chuỗi không phải từ tiếng Anh hoàn chỉnh (`decades` → `decad`), còn lemmatization phụ thuộc loại từ truyền vào (`went` → `go` khi chọn động từ; `mice` → `mouse` khi chọn danh từ). Đầu ra từ PDF phụ thuộc khả năng trích xuất văn bản; PDF chỉ chứa ảnh sẽ cần OCR.